# Flat BERT on the shared prepared split

Adapted from the supplied `standard-bert.ipynb`. It now reads the same prepared
train/validation/test files as the hierarchical model, chooses the best epoch
using **validation** Micro-F1, and evaluates the held-out test set afterward.
The original notebook selected epochs on its 604-post set named `test`; its
saved scores are historical development results, not independent test results.

Outputs and personal metadata are removed. Mixed-precision calls and scheduler
step counting are adjusted for the shared workflow. This adapted training has
not been executed. No earlier score is claimed for these changed settings.
Run `python scripts/prepare_experiment.py Dataset` first.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from the repository root; this also supports the default
# notebook-directory working directory without personal absolute paths.
if Path.cwd().name == "notebooks" and (Path.cwd().parent / "README.md").is_file():
    os.chdir(Path.cwd().parent)
print("Working directory: repository root" if (Path.cwd()/"README.md").is_file() else "Check working directory")


In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
import math
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

# ==========================================
# 0. 显存管理设置 (修正版)
# ==========================================
# 移除 expandable_segments，改用更稳定的碎片整理策略
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# ==========================================
# 1. 配置参数
# ==========================================
class CFG:
    model_name = "bert-base-uncased"
    max_len = 512       
    # 保持极低 Batch Size
    batch_size = 1      
    # 梯度累积
    accumulation_steps = 16 
    epochs = 10         
    lr = 2e-5           
    seed = 42           
    data_dir = "outputs_hier_strategy4" 
    out_dir = "outputs_flat_bert_shared" 

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG.seed)

if not os.path.exists(CFG.out_dir):
    os.makedirs(CFG.out_dir)

# ==========================================
# 2. 数据集处理 (Dataset)
# ==========================================
LABEL_COLUMNS = [
    "anger", "brain dysfunction (forget)", "emptiness", 
    "hopelessness", "loneliness", "sadness", 
    "suicide intent", "worthlessness"
]
NUM_LABELS = len(LABEL_COLUMNS)

class DepressionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True) 
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = self.df['text'].values
        self.labels = self.get_labels(self.df)

    def get_labels(self, df):
        label_matrix = np.zeros((len(df), NUM_LABELS))
        for idx, row in df.iterrows():
            emotions = row['emotions']
            if isinstance(emotions, list):
                for e in emotions:
                    if e in LABEL_COLUMNS:
                        label_idx = LABEL_COLUMNS.index(e)
                        label_matrix[idx, label_idx] = 1
        return label_matrix

    def __len__(self):
        return len(self.df)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.float)
        }

# ==========================================
# 3. 模型定义
# ==========================================
class FlatBERTModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super(FlatBERTModel, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output 
        output = self.dropout(pooled_output)
        logits = self.classifier(output)
        return logits

# ==========================================
# 4. 训练函数
# ==========================================
def train_fn(data_loader, model, optimizer, device, scheduler):
    model.train()
    final_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    scaler = GradScaler("cuda", enabled=(device.type == "cuda"))
    
    optimizer.zero_grad()
    
    # 使用 tqdm 显示进度
    progress_bar = tqdm(data_loader, total=len(data_loader), desc="Training")
    
    for step, data in enumerate(progress_bar):
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        labels = data['labels'].to(device)

        with autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            group_size = min(CFG.accumulation_steps, len(data_loader) - (step // CFG.accumulation_steps) * CFG.accumulation_steps)
            loss = loss / group_size 
        
        scaler.scale(loss).backward()
        
        final_loss += loss.item() * group_size
        
        if (step + 1) % CFG.accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
            # 释放显存
            del input_ids, attention_mask, labels, outputs, loss
    
    # 处理剩余的梯度
    if (step + 1) % CFG.accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

    torch.cuda.empty_cache()
    return final_loss / len(data_loader)

# ==========================================
# 5. 验证函数
# ==========================================
def eval_fn(data_loader, model, device):
    model.eval()
    final_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    fin_targets = []
    fin_outputs = []
    
    with torch.no_grad():
        for data in tqdm(data_loader, total=len(data_loader), desc="Validating"):
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_mask'].to(device)
            labels = data['labels'].to(device)

            with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                
            final_loss += loss.item()
            
            fin_targets.extend(labels.cpu().detach().numpy())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy())
            
            del input_ids, attention_mask, labels, outputs, loss
    
    torch.cuda.empty_cache()       
    return final_loss / len(data_loader), np.array(fin_outputs), np.array(fin_targets)

# ==========================================
# 6. 主程序
# ==========================================
if __name__ == "__main__":
    try:
        # 尝试清理之前的上下文
        torch.cuda.empty_cache()
        gc.collect()
        
        def load_shared_split(name):
            prepared = pd.read_json(os.path.join(CFG.data_dir, name + ".jsonl"), lines=True)
            return pd.DataFrame({
                "text": prepared["text_clean"],
                "emotions": prepared["y"].apply(lambda y: [lab for lab, bit in zip(LABEL_COLUMNS,y) if int(bit)==1])
            })
        train_df = load_shared_split("train")
        val_df = load_shared_split("val")
        test_df = load_shared_split("test")
        print(f"Shared split: train={len(train_df)}, validation={len(val_df)}, test={len(test_df)}")

        tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
        
        train_dataset = DepressionDataset(train_df, tokenizer, CFG.max_len)
        val_dataset = DepressionDataset(val_df, tokenizer, CFG.max_len)
        test_dataset = DepressionDataset(test_df, tokenizer, CFG.max_len)
        
        # num_workers=0 在 Windows/CUDA 上最稳定
        train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=0)
        test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=0)
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")
        
        model = FlatBERTModel(CFG.model_name, NUM_LABELS)
        model.to(device)
        
        param_optimizer = list(model.named_parameters())
        no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
        optimizer_parameters = [
            {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
            {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
        ]
        optimizer = AdamW(optimizer_parameters, lr=CFG.lr)
        
        total_steps = math.ceil(len(train_loader) / CFG.accumulation_steps) * CFG.epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=0, num_training_steps=total_steps
        )
        
        best_micro_f1 = 0
        print(f"Start Training Baseline (Safe Mode): {CFG.model_name}...")
        
        for epoch in range(CFG.epochs):
            train_loss = train_fn(train_loader, model, optimizer, device, scheduler)
            
            gc.collect()
            torch.cuda.empty_cache()
            
            val_loss, val_preds, val_targets = eval_fn(val_loader, model, device)
            
            val_preds_binary = (val_preds > 0.5).astype(int)
            
            micro_f1 = f1_score(val_targets, val_preds_binary, average='micro')
            macro_f1 = f1_score(val_targets, val_preds_binary, average='macro')
            
            print(f"Epoch {epoch+1}/{CFG.epochs} | Loss: {val_loss:.4f} | Micro-F1: {micro_f1:.4f} | Macro-F1: {macro_f1:.4f}")
            
            if micro_f1 > best_micro_f1:
                best_micro_f1 = micro_f1
                torch.save(model.state_dict(), os.path.join(CFG.out_dir, "best_standard_bert.pt"))
                best_metrics = {
                    "micro_f1": micro_f1,
                    "macro_f1": macro_f1,
                    "report": classification_report(val_targets, val_preds_binary, target_names=LABEL_COLUMNS, output_dict=True)
                }
                
        print("Best validation Micro-F1:", best_micro_f1)
        model.load_state_dict(torch.load(os.path.join(CFG.out_dir,"best_standard_bert.pt"),
                                         map_location=device, weights_only=True))
        test_loss, test_probs, test_targets = eval_fn(test_loader, model, device)
        test_predictions = (test_probs > 0.5).astype(int)
        best_metrics = {
            "micro_f1": f1_score(test_targets,test_predictions,average="micro",zero_division=0),
            "macro_f1": f1_score(test_targets,test_predictions,average="macro",zero_division=0),
            "report": classification_report(test_targets,test_predictions,target_names=LABEL_COLUMNS,
                                             output_dict=True,zero_division=0)
        }
        print("Final held-out test Micro-F1:",best_metrics["micro_f1"])
        print("Final held-out test Macro-F1:",best_metrics["macro_f1"])
        
        results = []
        report = best_metrics['report']
        model_tag = "Standard BERT (Truncated)"
        
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                results.append({
                    "Model": model_tag,
                    "Category": "Per-Label" if label in LABEL_COLUMNS else "Average",
                    "Label/Metric Name": label,
                    "Precision": metrics['precision'],
                    "Recall": metrics['recall'],
                    "F1-Score": metrics['f1-score'],
                    "Support": metrics['support']
                })
                
        df_res = pd.DataFrame(results)
        df_res.to_csv(os.path.join(CFG.out_dir, "standard_bert_performance.csv"), index=False)
        print(f"Results saved to {os.path.join(CFG.out_dir, 'standard_bert_performance.csv')}")

    except Exception as e:
        print(f"发生错误: {e}")
        print("建议：请尝试完全重启 Jupyter Notebook 或 电脑，然后再次运行。")